In [0]:
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("bronze_schema",   "bronze")
dbutils.widgets.text("silver_schema",   "silver")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
bronze_schema   = dbutils.widgets.get("bronze_schema")
silver_schema   = dbutils.widgets.get("silver_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
print(f"Silver pipeline — {catalog_name}")

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, split, explode, posexplode,
    regexp_extract, regexp_replace, to_date, concat_ws, md5, length,
    current_timestamp, current_date, coalesce, size, count as spark_count,
    row_number, monotonically_increasing_id, max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from functools import reduce
from datetime import datetime
 
def log_dqx(table_name, city, total, passed, failed):
    spark.sql(f"""
        INSERT INTO {catalog_name}.{metadata_schema}.dqx_execution_log VALUES (
            '{table_name}', '{city}', current_timestamp(),
            {total}, {passed}, {failed}, current_date()
        )
    """)

In [0]:
# Derive summary counts from silver tables (since Chicago & Dallas ran in separate notebooks)

chi_source_count   = spark.table(f"{catalog_name}.{bronze_schema}.bronze_chicago").count()
chi_passed         = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections").count()
chi_failed         = spark.table(f"{catalog_name}.{silver_schema}.quarantine_chicago").count()
chi_insp_count     = chi_passed
chi_viol_final     = spark.table(f"{catalog_name}.{silver_schema}.chicago_violations").count()
chi_no_viol_count  = spark.table(f"{catalog_name}.{silver_schema}.chicago_violations").filter("violation_code = 'NO_VIOLATION'").count()

dal_source_count       = spark.table(f"{catalog_name}.{bronze_schema}.bronze_dallas").count()
dal_passed             = spark.table(f"{catalog_name}.{silver_schema}.dallas_inspections").count()
dal_failed             = spark.table(f"{catalog_name}.{silver_schema}.quarantine_dallas").count()
dal_insp_count         = dal_passed
dal_viol_final         = spark.table(f"{catalog_name}.{silver_schema}.dallas_violations").count()
dal_no_viol_count      = spark.table(f"{catalog_name}.{silver_schema}.dallas_violations").filter("violation_code = 'NO_VIOLATION'").count()
dal_dupes_dropped      = "N/A (see 06b)"
dal_tiebreaker_dropped = "N/A (see 06b)"

print("Summary variables loaded from silver tables.")


In [0]:
 # PART C — UNIFIED SILVER VIEW
 # building vw_inspections_unified
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{silver_schema}.vw_inspections_unified AS
 
SELECT
    inspection_id,
    dba_name              AS establishment_name,
    aka_name,
    license_number,
    facility_type,
    risk                  AS risk_category,
    address,
    city,
    state,
    zip_code,
    inspection_date,
    inspection_type,
    results               AS inspection_result,
    violation_score       AS inspection_score,
    latitude,
    longitude,
    is_out_of_area,
    'Chicago'             AS source_city
FROM {catalog_name}.{silver_schema}.chicago_inspections
 
UNION ALL
 
SELECT
    inspection_id,
    restaurant_name       AS establishment_name,
    NULL                  AS aka_name,
    NULL                  AS license_number,
    'Food Establishment'  AS facility_type,
    NULL                  AS risk_category,
    street_address          AS address,
    'DALLAS'              AS city,
    'TX'                  AS state,
    zip_code,
    inspection_date,
    inspection_type,
    NULL                  AS inspection_result,
    inspection_score,
    latitude,
    longitude,
    is_out_of_area,
    'Dallas'              AS source_city
FROM {catalog_name}.{silver_schema}.dallas_inspections
""")
 
print("vw_inspections_unified created")

In [0]:
#building vw_violations_unified
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog_name}.{silver_schema}.vw_violations_unified AS
 
SELECT
    inspection_id,
    violation_code,
    violation_description,
    violation_detail,
    violation_memo,
    violation_points,
    'Chicago' AS source_city
FROM {catalog_name}.{silver_schema}.chicago_violations
 
UNION ALL
 
SELECT
    inspection_id,
    violation_code,
    violation_description,
    violation_detail,
    violation_memo,
    violation_points,
    'Dallas' AS source_city
FROM {catalog_name}.{silver_schema}.dallas_violations
""")
 
print("vw_violations_unified created")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ### C3. Verify unified views
 
# COMMAND ----------
 
print("=== INSPECTIONS UNIFIED ===")
display(spark.sql(f"""
    SELECT source_city, COUNT(*) AS inspections,
           SUM(CASE WHEN is_out_of_area THEN 1 ELSE 0 END) AS out_of_area_flagged
    FROM {catalog_name}.{silver_schema}.vw_inspections_unified
    GROUP BY source_city
"""))

In [0]:
print("=== VIOLATIONS UNIFIED ===")
display(spark.sql(f"""
    SELECT source_city, COUNT(*) AS violations,
           SUM(CASE WHEN violation_code = 'NO_VIOLATION' THEN 1 ELSE 0 END) AS no_violation_defaults
    FROM {catalog_name}.{silver_schema}.vw_violations_unified
    GROUP BY source_city
"""))

In [0]:
print("=" * 70)
print("SILVER LAYER SUMMARY")
print("=" * 70)
print(f"\nCHICAGO:")
print(f"  Bronze input:        {chi_source_count}")
print(f"  Passed DQX:          {chi_passed}")
print(f"  Quarantined:         {chi_failed}")
print(f"  Inspections:         {chi_insp_count}")
print(f"  Violations parsed:   {chi_viol_final}")
print(f"  Zero-viol defaults:  {chi_no_viol_count}")
print(f"\nDALLAS:")
print(f"  Bronze input:        {dal_source_count}")
print(f"  Full dupes dropped:  {dal_dupes_dropped}")
print(f"  Tiebreaker resolved: {dal_tiebreaker_dropped}")
print(f"  Passed DQX:          {dal_passed}")
print(f"  Quarantined:         {dal_failed}")
print(f"  Inspections:         {dal_insp_count}")
print(f"  Violations unpivoted:{dal_viol_final}")
print(f"  Zero-viol defaults:  {dal_no_viol_count}")

In [0]:
print("=== QUARANTINE: CHICAGO ===")
display(spark.sql(f"""
    SELECT _quarantine_reason, COUNT(*) AS cnt
    FROM {catalog_name}.{silver_schema}.quarantine_chicago
    GROUP BY _quarantine_reason ORDER BY cnt DESC
"""))
 
print("\n=== QUARANTINE: DALLAS ===")
display(spark.sql(f"""
    SELECT _quarantine_reason, COUNT(*) AS cnt
    FROM {catalog_name}.{silver_schema}.quarantine_dallas
    GROUP BY _quarantine_reason ORDER BY cnt DESC
"""))

In [0]:
print("=== DQX EXECUTION LOG ===")
display(spark.sql(f"""
    SELECT * FROM {catalog_name}.{metadata_schema}.dqx_execution_log
    ORDER BY execution_time DESC
"""))